## ComplexEntry: a self-rebuilding Torch model

A `ComplexEntry` is initialised from a `Manifest` plus a *constitution* — a Python source string defining exactly one function `f(manifest) -> payload`. Calling `.data` (or `.build()`) on the complex entry exec's the constitution and materialises its payload.

In this example the manifest holds:
- the **architecture hyper-parameters** of a small MLP (input / hidden / output sizes)
- every **weight tensor** keyed by its `state_dict` name

The constitution contains *everything* needed to rebuild the full `torch.nn.Module` — module class definition, architecture wiring, and `load_state_dict` — so the ComplexEntry can be restored from the manifest alone.


In [ ]:
import torch
import torch.nn as nn

import laila
from laila.policy.central.memory.schema import Manifest

### Step 1 — train a tiny model locally

In [ ]:
torch.manual_seed(0)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

arch = {"in_dim": 8, "hidden_dim": 16, "out_dim": 4}
model = MLP(**arch)

for _ in range(3):
    x = torch.randn(5, arch["in_dim"])
    loss = model(x).pow(2).sum()
    loss.backward()

probe = torch.randn(1, arch["in_dim"])
reference_output = model(probe).detach()
print("reference output:", reference_output)

### Step 2 — wrap weights and architecture into a Manifest

In [ ]:
arch_entry = laila.constant(data=arch, nickname="mlp.architecture")

weights = {
    name: laila.constant(data=tensor.detach().clone(), nickname=f"mlp.w.{name}")
    for name, tensor in model.state_dict().items()
}

manifest = Manifest(
    data={
        "architecture": arch_entry,
        "weights": weights,
    },
    nickname="mlp.manifest",
)

manifest.memorize().wait()
print("Manifest memorized with", sum(1 for _ in manifest), "referenced entries.")

### Step 3 — write the constitution source string

The constitution is a plain Python string. When the ComplexEntry builds, it is `exec`'d in a fresh namespace and the one function it defines is invoked with the manifest. Nothing else is available — the code must be self-contained.


In [ ]:
constitution_code = '''
import torch
import torch.nn as nn


class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


def build_mlp(manifest):
    recovered = manifest.realized
    arch = recovered["architecture"].data
    state_dict = {
        name: entry.data for name, entry in recovered["weights"].items()
    }
    model = MLP(**arch)
    model.load_state_dict(state_dict)
    model.eval()
    return model
'''


### Step 4 — create the ComplexEntry and materialise it

Note that after the manifest has been memorized, we can throw the original `model` object away — the ComplexEntry will rebuild it from just the constitution string and the manifest.


In [ ]:
del model

complex_model = laila.variable(
    manifest=manifest,
    constitution=constitution_code,
    nickname="mlp.complex",
)

print("Before build, state:", complex_model.state)

rebuilt = complex_model.data

print("After build, state: ", complex_model.state)
print("Rebuilt type:       ", type(rebuilt).__name__)

### Step 5 — verify the rebuilt model matches the original

In [ ]:
rebuilt_output = rebuilt(probe).detach()

print("reference output:", reference_output)
print("rebuilt   output:", rebuilt_output)
print("match:           ", torch.allclose(reference_output, rebuilt_output))

### Clean up

In [ ]:
manifest.forget().wait()
print("Cleanup done.")